# ETL Process for Data Warehouse Project 1

First, import dependencies such as pandas for efficient and easy data processing.

In [255]:
import pandas as pd

Then, read in the first data set `Airline Dataset Updated - v2.csv` which I have renamed to `booking` since each row contains information for a booking between a passenger and the airline.

In [256]:
booking = pd.read_csv('bookings.csv')

booking.head()

,Passenger ID,First Name,Last Name,Gender,Age,Nationality,Airport Name,Airport Country Code,Country Name,Airport Continent,Continents,Departure Date,Arrival Airport,Pilot Name,Flight Status
0,ABVWIg,Edithe,Leggis,Female,62,Japan,Coldfoot Airport,US,United States,NAM,North America,6/28/2022,CXF,Fransisco Hazeldine,On Time
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua,Kugluktuk Airport,CA,Canada,NAM,North America,12/26/2022,YCO,Marla Parsonage,On Time
2,CdUz2g,Darby,Felgate,Male,67,Russia,Grenoble-Isère Airport,FR,France,EU,Europe,1/18/2022,GNB,Rhonda Amber,On Time
3,BRS38V,Dominica,Pyle,Female,71,China,Ottawa / Gatineau Airport,CA,Canada,NAM,North America,9/16/2022,YND,Kacie Commucci,Delayed
4,9kvTLo,Bay,Pencost,Male,21,China,Gillespie Field,US,United States,NAM,North America,2/25/2022,SEE,Ebonee Tree,On Time


Next the data is cleaned of null or invalid data.

In [257]:
# Check for null data.
booking.info()

# Check for invalid airport codes.
invalid_airports = booking[~booking['Arrival Airport'].str.match(r'^[A-Z]{3}$')]

# Drop invalid rows.
booking.drop(invalid_airports.index, inplace=True)

<class 'pandas.DataFrame'>
RangeIndex: 98619 entries, 0 to 98618
Data columns (total 15 columns):
 #   Column                Non-Null Count  Dtype
---  ------                --------------  -----
 0   Passenger ID          98619 non-null  str  
 1   First Name            98619 non-null  str  
 2   Last Name             98619 non-null  str  
 3   Gender                98619 non-null  str  
 4   Age                   98619 non-null  int64
 5   Nationality           98619 non-null  str  
 6   Airport Name          98619 non-null  str  
 7   Airport Country Code  98619 non-null  str  
 8   Country Name          98619 non-null  str  
 9   Airport Continent     98619 non-null  str  
 10  Continents            98619 non-null  str  
 11  Departure Date        98619 non-null  str  
 12  Arrival Airport       98619 non-null  str  
 13  Pilot Name            98619 non-null  str  
 14  Flight Status         98619 non-null  str  
dtypes: int64(1), str(14)
memory usage: 11.3 MB


The invalid rows are dropped. Next, the dimension and fact tables will be created, starting with the passenger dimension table.

In [258]:
# Select columns
dim_passenger = booking[['Passenger ID', 'First Name', 'Last Name', 'Gender', 'Age', 'Nationality']].copy()

# Check for duplicate rows
print(dim_passenger.duplicated(subset=['Passenger ID']).sum())

# Rename columns
dim_passenger.rename(columns={"Passenger ID": "PassengerID", "First Name": "FName", "Last Name": "LName"}, inplace=True)

# Write to csv
dim_passenger.to_csv('dim_passenger.csv', index=False)

dim_passenger

0


,PassengerID,FName,LName,Gender,Age,Nationality
0,ABVWIg,Edithe,Leggis,Female,62,Japan
1,jkXXAX,Elwood,Catt,Male,62,Nicaragua
2,CdUz2g,Darby,Felgate,Male,67,Russia
3,BRS38V,Dominica,Pyle,Female,71,China
4,9kvTLo,Bay,Pencost,Male,21,China
...,...,...,...,...,...,...
98614,hnGQ62,Gareth,Mugford,Male,85,China
98615,2omEzh,Kasey,Benedict,Female,19,Russia
98616,VUPiVG,Darrin,Lucken,Male,65,Indonesia
98617,E47NtS,Gayle,Lievesley,Female,34,China


There appears to be no duplicate passengers since every passenger ID is unique. So the desired columns are easily selected and written to a csv file. 

Next, I will create the airport table which will be extended later with another dataset.

In [259]:
# Select desired columns
dim_airport = booking[['Arrival Airport', 'Airport Name', 'Country Name', 'Continents']].copy()

# Rename columns for interpretability
dim_airport.rename(columns={"Arrival Airport":"IATA", "Airport Name":"Name", "Country Name":"Country", "Continents":"Continent"}, inplace=True)

# Check for duplicates
print(dim_airport.duplicated(subset=['IATA']).sum())

# Drop duplicates
dim_airport.drop_duplicates(subset=['IATA'], inplace=True)

dim_airport


88675


,IATA,Name,Country,Continent
0,CXF,Coldfoot Airport,United States,North America
1,YCO,Kugluktuk Airport,Canada,North America
2,GNB,Grenoble-Isère Airport,France,Europe
3,YND,Ottawa / Gatineau Airport,Canada,North America
4,SEE,Gillespie Field,United States,North America
...,...,...,...,...
68145,HDN,Yampa Valley Airport,United States,North America
71596,HIJ,Hiroshima Airport,Japan,Asia
73571,CXM,Camaxilo Airport,Angola,Africa
78509,ROR,Babelthuap Airport,Palau,Oceania


As explained in the task sheet, `Arrival Airport` is a mislabelling of the IATA code of the departure airport so it was renamed to reflect this. The other columns were also renamed to reflect better naming conventions in relational databases.

Next, any duplicate airports were dropped since they only need to be listed once.

Now, the pilot table will be made.

In [260]:
# Select desired columns.
dim_pilot = booking[['Pilot Name']].copy()

# Check for duplicates
print(dim_pilot.duplicated(subset=['Pilot Name']).sum())

# Drop duplicates
dim_pilot.drop_duplicates(subset=['Pilot Name'], inplace=True)

# Create PilotID and add to original table
dim_pilot.index.name = 'PilotID'
dim_pilot = dim_pilot.reset_index()
booking = booking.merge(dim_pilot[['PilotID', 'Pilot Name']], on='Pilot Name', how='left')

# Separate name columns
dim_pilot["FName"] = dim_pilot['Pilot Name'].str.split(" ").str[:-1].str.join(" ")
dim_pilot["LName"] = dim_pilot['Pilot Name'].str.split(" ").str[-1]
dim_pilot.drop(columns=["Pilot Name"], inplace=True)

# Write to csv
dim_pilot.to_csv("dim_pilot.csv", index=False)

dim_pilot

13


,PilotID,FName,LName
0,0,Fransisco,Hazeldine
1,1,Marla,Parsonage
2,2,Rhonda,Amber
3,3,Kacie,Commucci
4,4,Ebonee,Tree
...,...,...,...
97675,98614,Pammie,Kingscote
97676,98615,Dorice,Lochran
97677,98616,Gearalt,Main
97678,98617,Judon,Chasle


The assumption is used that any pilots with the same name, are the same pilot; any duplicates are dropped.

Then each pilot is given a `PilotID` which is merged back into the original table so the fact table can be easily created.

Then, `Pilot Name` is divided into `LName` and `FName` to ensure first normal form and the data is written to csv.

Next, the fact table will be created.

In [261]:
# Take desired columns
fact_booking = booking[['Passenger ID', 'Arrival Airport', 'PilotID', 'Flight Status', 'Departure Date']].copy()

# Change all dates to the same format
booking['Departure Date'] = booking['Departure Date'].str.replace('/', '-')

# Create BookingID
fact_booking.index.name = 'BookingID'
fact_booking = fact_booking.reset_index()

# Print to csv
fact_booking.to_csv("fact_booking.csv", index=False)

fact_booking

,BookingID,Passenger ID,Arrival Airport,PilotID,Flight Status,Departure Date
0,0,ABVWIg,CXF,0,On Time,6/28/2022
1,1,jkXXAX,YCO,1,On Time,12/26/2022
2,2,CdUz2g,GNB,2,On Time,1/18/2022
3,3,BRS38V,YND,3,Delayed,9/16/2022
4,4,9kvTLo,SEE,4,On Time,2/25/2022
...,...,...,...,...,...,...
97688,97688,hnGQ62,HAA,98614,Cancelled,12-11-2022
97689,97689,2omEzh,IVA,98615,Cancelled,10/30/2022
97690,97690,VUPiVG,ABC,98616,On Time,09-10-2022
97691,97691,E47NtS,GGN,98617,Cancelled,10/26/2022


Same dates use `/` where as others use `-`, so all were changed to use `-`. Then each record was given a BookingID and the table was written to csv.

Now the `airport` dataset will be imported.

In [262]:
airport = pd.read_csv('airports.csv')

airport.head()

,id,ident,type,name,latitude_deg,longitude_deg,elevation_ft,continent,iso_country,iso_region,municipality,scheduled_service,icao_code,iata_code,gps_code,local_code,home_link,wikipedia_link,keywords
0,6523,00A,heliport,Total RF Heliport,40.070985,-74.933689,11.0,NaN,US,US-PA,Bensalem,no,NaN,NaN,K00A,00A,https://www.penndot.pa.gov/TravelInPA/airports...,NaN,NaN
1,323361,00AA,small_airport,Aero B Ranch Airport,38.704022,-101.473911,3435.0,NaN,US,US-KS,Leoti,no,NaN,NaN,00AA,00AA,NaN,NaN,NaN
2,6524,00AK,small_airport,Lowell Field,59.947733,-151.692524,450.0,NaN,US,US-AK,Anchor Point,no,NaN,NaN,00AK,00AK,NaN,NaN,NaN
3,6525,00AL,small_airport,Epps Airpark,34.864799,-86.770302,820.0,NaN,US,US-AL,Harvest,no,NaN,NaN,00AL,00AL,NaN,NaN,NaN
4,506791,00AN,small_airport,Katmai Lodge Airport,59.093287,-156.456699,80.0,NaN,US,US-AK,King Salmon,no,NaN,NaN,00AN,00AN,NaN,NaN,NaN


In [263]:
airport = airport[['type', 'name', 'latitude_deg', 'longitude_deg', 'elevation_ft', 'iso_region', 'municipality', 'iata_code']].copy()

# --- STEP 1: Primary Merge (IATA) ---
# Drop duplicates in airport_info based on IATA before merging!
step1 = dim_airport.merge(
    airport.drop_duplicates(subset=['iata_code']), # <-- THE FIX
    left_on='IATA', 
    right_on='iata_code', 
    how='left', 
    indicator=True
)

# --- STEP 2: Split Successes and Failures ---
matched_on_iata = step1[step1['_merge'] == 'both'].drop(columns=['_merge'])
failed_rows = step1[step1['_merge'] == 'left_only']
clean_failed_rows = failed_rows[dim_airport.columns] 

# --- STEP 3: Backup Merge (Name) ---
# Drop duplicates in airport_info based on Name before merging!
matched_on_name = clean_failed_rows.merge(
    airport.drop_duplicates(subset=['name']), # <-- THE FIX
    left_on='Name', 
    right_on='name', 
    how='left'
)

# --- STEP 4: Stack everything back together ---
final_airport = pd.concat([matched_on_iata, matched_on_name], ignore_index=True)
final_airport = final_airport[['IATA', 'Name', 'type', 'Continent', 'Country', 'iso_region', 'municipality', 'latitude_deg', 'longitude_deg', 'elevation_ft']]
final_airport.rename(columns={"type": "Type", "iso_region": "Region", "municipality": "Municipality", "latitude_deg": "Latitude", "longitude_deg": "Longitude", "elevation_ft": "Elevation"}, inplace=True)

final_airport.to_csv("dim_airport.csv", index=False)